# Decision Trees and Random Forests

## What you'll learn
- How a decision tree makes predictions with a series of yes/no questions
- How to print a tree's rules and limit its depth
- How a random forest combines many trees for better predictions
- Feature importance: which columns the model relies on most
- That trees work for regression too

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

In [ ]:
from sklearn.model_selection import train_test_split

X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## A decision tree

A **decision tree** asks one question at a time ("is days to pay above 42?") and follows
the answer down a branch, until it reaches a leaf with a prediction. Trees do not need
scaled features, and their rules are easy to read. `max_depth` limits how many questions
deep the tree can go.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(f"Test accuracy: {accuracy_score(y_test, tree.predict(X_test)):.1%}\n")
print(export_text(tree, feature_names=list(X.columns)))

Each line is a question; `class: 1` means the leaf predicts "late". You could hand these
rules to the collections team as they are.

## Too deep: overfitting

A tree with no depth limit keeps splitting until it has memorised the training data. It
then looks perfect on training data and worse on new data. That gap is **overfitting**.

In [ ]:
for depth in [2, 3, 5, 8, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth={str(depth):4s} train {accuracy_score(y_train, t.predict(X_train)):.1%}"
          f"  test {accuracy_score(y_test, t.predict(X_test)):.1%}")

## A random forest

A **random forest** trains many trees, each on a random sample of rows and features, and
lets them vote. Individual trees make different mistakes, so the vote is usually more
accurate and less prone to overfitting than any single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
forest.fit(X_train, y_train)
print(f"Random forest test accuracy: {accuracy_score(y_test, forest.predict(X_test)):.1%}")

## Feature importance

Forests report how much each feature helped their decisions. It is a quick way to see what
drives the prediction (it shows association, not cause).

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
importance.tail(10).plot(kind="barh", figsize=(7, 4), title="Top 10 features for late payment")
plt.xlabel("Importance")
plt.show()

## Trees for regression

The same ideas predict numbers: `RandomForestRegressor` works like `LinearRegression`,
with `fit` and `predict`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

Xr = df[NUMERIC]
yr = df["next_year_spend"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(Xr_train, yr_train)
print(f"Random forest regression R2: {r2_score(yr_test, reg.predict(Xr_test)):.3f}")

## Practice

1. Train a tree with `max_depth=2` and print its rules. What is the first question it asks?
2. Train forests with `n_estimators` of 10, 100, and 500. How does test accuracy change?
3. Which three features matter most to the forest? Do they match the logistic regression
   coefficients from lesson 3?
4. Compare the regression forest's R squared with linear regression on the same features.

In [ ]:
# Your practice code here

## Summary

- A decision tree predicts with a chain of yes/no questions; its rules are readable.
- Deep trees overfit: great on training data, worse on test data. Limit `max_depth`.
- A random forest averages many trees and is usually more accurate.
- `feature_importances_` shows which features the forest relies on.
- Trees and forests handle both classification and regression.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score

X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1.
small = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
print(export_text(small, feature_names=list(X.columns)))

# 2.
for n in [10, 100, 500]:
    f = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, random_state=42, n_jobs=-1)
    f.fit(X_train, y_train)
    print(n, "trees:", f"{accuracy_score(y_test, f.predict(X_test)):.1%}")

# 3.
print(pd.Series(f.feature_importances_, index=X.columns).sort_values(ascending=False).head(3))

# 4.
Xr, yr = df[NUMERIC], df["next_year_spend"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
for name, m in [("forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
                ("linear", LinearRegression())]:
    m.fit(Xr_train, yr_train)
    print(name, f"R2 {r2_score(yr_test, m.predict(Xr_test)):.3f}")